# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 3.0.2


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string
activity_year    string
lei              string
county_code      string
action_taken     string
loan_amount      string
income           string
dtype: object


  source_row activity_year  ... loan_amount income
0          2          2023  ...        oops   72.0
1          3          2023  ...    585000.0  174.0
2          5          2023  ...    275000.0    BAD
3          7          2023  ...    295000.0   63.0
4          8          2023  ...   1125000.0  247.0

[5 rows x 7 columns]

## H1. Собственная функция очистки

In [3]:
def clean_export(source):
    """Готовит учебную выгрузку: возвращает (clean, quarantine, audit).

    source     - таблица, прочитанная с dtype="string" (исходный текст); не изменяется.
    clean      - строки, прошедшие обязательные правила; числовые поля - числа.
    quarantine - отклонённые строки с ИСХОДНЫМИ значениями и столбцом reason.
    audit      - словарь счётчиков: input, replays, clean, quarantine, parse_errors.
    Останавливается (AssertionError), если у одного известного source_row
    есть две разные версии записи.
    """
    data = source.copy(deep=True)

    # 1. Повторные доставки: сначала проверяем конфликт версий, потом исключаем
    known_key = data["source_row"].notna()
    distinct_known = data.loc[known_key].drop_duplicates()
    assert distinct_known["source_row"].is_unique, "Конфликт: разные версии одного source_row"
    replay_mask = known_key & data["source_row"].duplicated()
    replays = int(replay_mask.sum())
    data = data.loc[~replay_mask].copy()

    # 2. Внешние пробелы в коде территории
    data["county_code"] = data["county_code"].str.strip()

    # 3. Числовые преобразования; считаем только НОВЫЕ ошибки (было значение -> стал пропуск)
    parse_errors = {}
    for field in ["source_row", "loan_amount", "income", "action_taken"]:
        numbers = pd.to_numeric(data[field], errors="coerce")
        parse_errors[field] = int((data[field].notna() & numbers.isna()).sum())
        data[field] = numbers

    # 4. Обязательные правила; income в них не входит - он может отсутствовать
    bad_key = data["source_row"].isna()
    bad_amount = data["loan_amount"].isna() | data["loan_amount"].le(0).fillna(False)
    bad_outcome = ~data["action_taken"].isin([1, 2, 3])
    bad_county = ~data["county_code"].eq("11001").fillna(False)

    reason = pd.Series("", index=data.index, dtype="string")
    reason.loc[bad_key.to_numpy(dtype=bool)] += "unknown_source_row;"
    reason.loc[bad_amount.to_numpy(dtype=bool)] += "invalid_loan_amount;"
    reason.loc[bad_outcome.to_numpy(dtype=bool)] += "invalid_action_taken;"
    reason.loc[bad_county.to_numpy(dtype=bool)] += "invalid_county;"
    invalid = reason.ne("").to_numpy(dtype=bool)

    # 5. Подготовленный слой и карантин (в карантине - исходный текст + причина)
    clean = data.loc[~invalid].copy()
    quarantine = source.loc[data.index[invalid]].copy()
    quarantine["reason"] = reason.loc[invalid]

    # 6. Журнал и баланс
    audit = {
        "input": len(source),
        "replays": replays,
        "clean": len(clean),
        "quarantine": len(quarantine),
        "parse_errors": parse_errors,
    }
    assert audit["input"] == audit["replays"] + audit["clean"] + audit["quarantine"], "Баланс не сошёлся"
    return clean, quarantine, audit

## H2. Год и известный доход

In [5]:
clean, quarantine, audit = clean_export(export)
print(audit)
quarantine

{'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': {'source_row': 0, 'loan_amount': 1, 'income': 1, 'action_taken': 0}}


  source_row activity_year  ... income                 reason
0          2          2023  ...   72.0   invalid_loan_amount;
1          3          2023  ...  174.0  invalid_action_taken;

[2 rows x 8 columns]

In [6]:
# activity_year в подготовленных строках - строка "2023"
assert clean["activity_year"].eq("2023").all()

total = len(clean)
income_known_mask = clean["income"].notna()
income_known = int(income_known_mask.sum())
income_unknown = int(clean["income"].isna().sum())
assert income_known + income_unknown == total

# среднее - только по известным значениям; пропуск нулём не заменяем
mean_income = clean.loc[income_known_mask, "income"].mean()

print("Строк в clean:", total)
print("income известен:", income_known)
print("income неизвестен:", income_unknown)
print(f"Средний известный доход: {mean_income:.2f} (по {income_known} значениям)")
print("Единица: тысячи USD в год")

Строк в clean: 6828
income известен: 6485
income неизвестен: 343
Средний известный доход: 237.07 (по 6485 значениям)
Единица: тысячи USD в год


In [7]:
# Баланс и сохранность входа
assert len(export) == len(clean) + len(quarantine) + audit["replays"]
assert_frame_equal(export, raw_copy)
print(len(export), "=", len(clean), "+", len(quarantine), "+", audit["replays"])
print("export не изменился")

6832 = 6828 + 2 + 2
export не изменился


## H3. Проверка отчётного года

In [9]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"

year_ok = wrong_year["activity_year"].eq("2023").fillna(False).all()
print("Год у всех строк 2023:", year_ok)
assert not year_ok  # проверка обнаружила нарушение

# контроль: исходный export проверку проходит и не изменился
assert export["activity_year"].eq("2023").fillna(False).all()
assert_frame_equal(export, raw_copy)

Год у всех строк 2023: False


In [10]:
# Три разных вида проблем на данных занятия
income_missing = int(export["income"].isna().sum())
print("1. Исходные пропуски income:", income_missing)

income_numbers = pd.to_numeric(export["income"], errors="coerce")
income_error = export["income"].notna() & income_numbers.isna()
print("2. Новые ошибки преобразования income:", int(income_error.sum()))
print(export.loc[income_error, ["source_row", "income"]])

loan_numbers = pd.to_numeric(export["loan_amount"], errors="coerce")
loan_error = export["loan_amount"].notna() & loan_numbers.isna()
print("   Новые ошибки преобразования loan_amount:", int(loan_error.sum()))
print(export.loc[loan_error, ["source_row", "loan_amount"]])

q1, q3 = clean["loan_amount"].quantile([0.25, 0.75])
threshold = q3 + 1.5 * (q3 - q1)
unusual = clean["loan_amount"].gt(threshold)
print("3. Необычно большие суммы: Q1 =", q1, "Q3 =", q3, "порог =", threshold)
print("   Строк выше порога:", int(unusual.sum()), "; максимум:", clean["loan_amount"].max())
print(clean.loc[clean["loan_amount"].idxmax(), ["source_row", "loan_amount", "income"]])

1. Исходные пропуски income: 342
2. Новые ошибки преобразования income: 1
  source_row income
2          5    BAD
   Новые ошибки преобразования loan_amount: 1
  source_row loan_amount
0          2        oops
3. Необычно большие суммы: Q1 = 315000.0 Q3 = 755000.0 порог = 1415000.0
   Строк выше порога: 355 ; максимум: 57005000.0
source_row          326.0
loan_amount    57005000.0
income               <NA>
Name: 249, dtype: Float64


In [11]:
# Где именно сработает проверка: строка 0 (source_row 2) имеет oops в сумме
c_w, q_w, a_w = clean_export(wrong_year)
print("В clean все годы 2023:", c_w["activity_year"].eq("2023").all())
print("В карантине год 2022 у строк:", int(q_w["activity_year"].eq("2022").sum()))

В clean все годы 2023: True
В карантине год 2022 у строк: 1


## H+1. Конфликт исходных значений

In [13]:
conflict = export.copy(deep=True)
replay_flag = conflict["source_row"].notna() & conflict["source_row"].duplicated()
last_replay = replay_flag[replay_flag].index[-1]
print("Последняя повторная доставка: индекс", last_replay,
      "source_row", conflict.loc[last_replay, "source_row"],
      "loan_amount", conflict.loc[last_replay, "loan_amount"])
conflict.loc[last_replay, "loan_amount"] = "999999.0"

try:
    clean_export(conflict)
except AssertionError as err:
    print("Ожидаемая остановка:", err)
else:
    raise AssertionError("Ошибка проверки: конфликт версий не обнаружен")

assert_frame_equal(export, raw_copy)

Последняя повторная доставка: индекс 6831 source_row 16 loan_amount 305000.0
Ожидаемая остановка: Конфликт: разные версии одного source_row


## H+2. Совпадение опубликованных полей

In [15]:
fields_without_row = [c for c in decisions.columns if c != "source_row"]
dedup = decisions.drop_duplicates(subset=fields_without_row)

before_n, after_n = len(decisions), len(dedup)
before_sum, after_sum = decisions["loan_amount"].sum(), dedup["loan_amount"].sum()

print("Поля сравнения:", fields_without_row)
print("Строк до:", before_n, "после:", after_n, "разница:", before_n - after_n)
print("Сумма loan_amount до:", before_sum, "после:", after_sum, "разница:", before_sum - after_sum)

Поля сравнения: ['activity_year', 'lei', 'county_code', 'action_taken', 'loan_amount', 'income']
Строк до: 6830 после: 6757 разница: 73
Сумма loan_amount до: 4577460000.0 после: 4547855000.0 разница: 29605000.0
